# EDA dan Wordcloud ABSA Hotel Santika

Notebook ini dibuat untuk Google Colab. Upload dua folder berikut ke Google Drive:

- `Data Merge`, berisi `dataset_absa_santika_merged_raw.csv`
- `Data Preprocessing`, berisi `dataset_absa_santika_clean_v2.csv`

Struktur yang didukung otomatis:

```text
MyDrive/ABSA Hotel Santika/Data Merge/...
MyDrive/ABSA Hotel Santika/Data Preprocessing/...
```

atau:

```text
MyDrive/Data Merge/...
MyDrive/Data Preprocessing/...
```

Jika folder kamu berbeda, ubah variabel `PROJECT_DIR`, `MANUAL_DATA_MERGE_DIR`, atau `MANUAL_DATA_PREPROCESSING_DIR` pada cell konfigurasi.

In [ ]:
# Install library tambahan. Jalankan cell ini sekali di awal runtime Colab.
!pip -q install wordcloud Sastrawi

import re
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown
from wordcloud import WordCloud, STOPWORDS

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titleweight"] = "bold"
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)


## 1. Mount Google Drive dan cari folder data

In [ ]:
# Mount Google Drive.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Tidak berjalan di Colab atau Drive sudah termount:", exc)

MYDRIVE = Path("/content/drive/MyDrive")

# Opsi 1: kalau kamu upload folder parent bernama ABSA Hotel Santika.
PROJECT_DIR = MYDRIVE / "ABSA Hotel Santika"

# Opsi 2: kalau lokasi folder berbeda, isi manual seperti contoh berikut:
# MANUAL_DATA_MERGE_DIR = "/content/drive/MyDrive/Nama Folder Kamu/Data Merge"
# MANUAL_DATA_PREPROCESSING_DIR = "/content/drive/MyDrive/Nama Folder Kamu/Data Preprocessing"
MANUAL_DATA_MERGE_DIR = None
MANUAL_DATA_PREPROCESSING_DIR = None

def resolve_folder(folder_name, manual_path=None):
    if manual_path:
        path = Path(manual_path)
        if not path.exists():
            raise FileNotFoundError(f"Folder manual tidak ditemukan: {path}")
        return path

    candidates = [
        PROJECT_DIR / folder_name,
        MYDRIVE / folder_name,
        MYDRIVE / "Colab Notebooks" / "ABSA Hotel Santika" / folder_name,
    ]
    for path in candidates:
        if path.exists() and path.is_dir():
            return path

    print(f"Folder '{folder_name}' belum ditemukan di kandidat umum. Mencari di MyDrive, mungkin agak lama...")
    matches = [p for p in MYDRIVE.rglob(folder_name) if p.is_dir()]
    if not matches:
        raise FileNotFoundError(
            f"Folder '{folder_name}' tidak ditemukan. Cek nama folder atau isi MANUAL_*_DIR."
        )
    return matches[0]

DATA_MERGE_DIR = resolve_folder("Data Merge", MANUAL_DATA_MERGE_DIR)
DATA_PREPROCESSING_DIR = resolve_folder("Data Preprocessing", MANUAL_DATA_PREPROCESSING_DIR)
OUTPUT_DIR = DATA_PREPROCESSING_DIR.parent / "EDA_Wordcloud_Output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data Merge folder       :", DATA_MERGE_DIR)
print("Data Preprocessing folder:", DATA_PREPROCESSING_DIR)
print("Output folder           :", OUTPUT_DIR)


## 2. Load CSV raw merge dan clean preprocessing

In [ ]:
def pick_csv(folder, preferred_names):
    csv_files = sorted(folder.glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"Tidak ada file CSV di folder: {folder}")

    lower_map = {p.name.lower(): p for p in csv_files}
    for name in preferred_names:
        if name.lower() in lower_map:
            return lower_map[name.lower()]

    print("CSV yang tersedia:")
    for p in csv_files:
        print("-", p.name)
    print("Memakai CSV pertama:", csv_files[0].name)
    return csv_files[0]

def read_csv_safely(path):
    encodings = ["utf-8-sig", "utf-8", "latin1"]
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception as exc:
            last_error = exc
    raise last_error

RAW_CSV = pick_csv(DATA_MERGE_DIR, ["dataset_absa_santika_merged_raw.csv"])
CLEAN_CSV = pick_csv(DATA_PREPROCESSING_DIR, ["dataset_absa_santika_clean_v2.csv"])

raw_df = read_csv_safely(RAW_CSV)
clean_df = read_csv_safely(CLEAN_CSV)

file_info = pd.DataFrame([
    {"dataset": "raw_merge", "file": RAW_CSV.name, "rows": len(raw_df), "columns": len(raw_df.columns)},
    {"dataset": "clean_preprocessing", "file": CLEAN_CSV.name, "rows": len(clean_df), "columns": len(clean_df.columns)},
])
display(file_info)

print("Kolom raw merge:")
print(list(raw_df.columns))
print("\nKolom clean preprocessing:")
print(list(clean_df.columns))


## 3. Standarisasi kolom agar raw dan clean bisa dibandingkan

In [ ]:
COLUMN_CANDIDATES = {
    "review_id": ["ID_Review", "review_id", "id_review", "id"],
    "platform": ["Platform", "platform", "source", "ota"],
    "hotel_name": ["Nama_Hotel", "hotel_name", "nama_hotel", "hotel"],
    "review_date": ["Review_Date", "date", "tanggal", "review_date"],
    "text_review": ["Text_Review", "text_review", "review", "text", "ulasan"],
    "text_original": ["text_review_original", "Text_Review_Original", "original_text", "text_asli"],
}

def normalize_name(value):
    return re.sub(r"[^a-z0-9]", "", str(value).strip().lower())

def detect_col(df, candidates):
    exact = {str(c).strip().lower(): c for c in df.columns}
    compact = {normalize_name(c): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in exact:
            return exact[cand.lower()]
        if normalize_name(cand) in compact:
            return compact[normalize_name(cand)]
    return None

def parse_dates_safely(series):
    dt1 = pd.to_datetime(series, errors="coerce")
    dt2 = pd.to_datetime(series, errors="coerce", dayfirst=True)
    return dt1.where(dt1.notna(), dt2)

def standardize_reviews(df, dataset_name):
    detected = {key: detect_col(df, cands) for key, cands in COLUMN_CANDIDATES.items()}
    print(f"Kolom terdeteksi untuk {dataset_name}:")
    for key, col in detected.items():
        print(f"- {key:12s}: {col}")

    out = pd.DataFrame(index=df.index)
    for key in ["review_id", "platform", "hotel_name", "review_date", "text_review", "text_original"]:
        col = detected.get(key)
        out[key] = df[col] if col is not None else np.nan

    out["dataset"] = dataset_name
    out["review_id"] = out["review_id"].astype(str).str.strip()
    out["platform"] = out["platform"].fillna("").astype(str).str.strip()
    out["hotel_name"] = out["hotel_name"].fillna("").astype(str).str.strip()
    out["review_date"] = out["review_date"].fillna("").astype(str).str.strip()
    out["text_review"] = out["text_review"].fillna("").astype(str).str.strip()
    out["text_original"] = out["text_original"].fillna("").astype(str).str.strip()
    out.loc[out["text_review"].str.lower().isin(["nan", "none", "null"]), "text_review"] = ""
    out["review_date_parsed"] = parse_dates_safely(out["review_date"])
    out["char_len"] = out["text_review"].str.len()
    out["word_count"] = out["text_review"].str.split().apply(len)
    return out

raw_std = standardize_reviews(raw_df, "raw_merge")
print()
clean_std = standardize_reviews(clean_df, "clean_preprocessing")

display(raw_std.head(3))
display(clean_std.head(3))


## 4. EDA ringkas per dataset

In [ ]:
def dataset_summary(name, df):
    nonempty_text = df["text_review"].str.strip().ne("")
    parsed_dates = df["review_date_parsed"].notna()
    return {
        "dataset": name,
        "rows": len(df),
        "unique_review_id": df["review_id"].replace("", np.nan).nunique(),
        "duplicate_review_id": df["review_id"].replace("", np.nan).duplicated().sum(),
        "nonempty_text": int(nonempty_text.sum()),
        "empty_text": int((~nonempty_text).sum()),
        "duplicate_text": int(df.loc[nonempty_text, "text_review"].duplicated().sum()),
        "platform_count": df["platform"].replace("", np.nan).nunique(),
        "hotel_count": df["hotel_name"].replace("", np.nan).nunique(),
        "date_parsed": int(parsed_dates.sum()),
        "min_date": df.loc[parsed_dates, "review_date_parsed"].min(),
        "max_date": df.loc[parsed_dates, "review_date_parsed"].max(),
        "avg_char_len": round(df.loc[nonempty_text, "char_len"].mean(), 2),
        "median_char_len": round(df.loc[nonempty_text, "char_len"].median(), 2),
        "avg_word_count": round(df.loc[nonempty_text, "word_count"].mean(), 2),
        "median_word_count": round(df.loc[nonempty_text, "word_count"].median(), 2),
    }

summary_df = pd.DataFrame([
    dataset_summary("raw_merge", raw_std),
    dataset_summary("clean_preprocessing", clean_std),
])
display(summary_df)

def missing_table(df, title):
    miss = pd.DataFrame({
        "column": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_pct": (df.isna().mean().values * 100).round(2),
    }).sort_values("missing_pct", ascending=False)
    display(Markdown(f"### Missing values: {title}"))
    display(miss.head(20))

missing_table(raw_df, "Raw Merge")
missing_table(clean_df, "Clean Preprocessing")


## 5. Visualisasi distribusi platform, hotel, waktu, dan panjang teks

In [ ]:
def save_current_figure(filename):
    if filename:
        path = OUTPUT_DIR / filename
        plt.savefig(path, dpi=180, bbox_inches="tight")
        print(f"Tersimpan: {path}")

def plot_top_counts(df, col, title, top_n=10, save_as=None):
    vc = df[col].replace("", np.nan).dropna().value_counts().head(top_n)
    if vc.empty:
        print(f"Tidak ada data untuk {title}")
        return pd.DataFrame(columns=[col, "count", "percent"])
    plot_df = vc.reset_index()
    plot_df.columns = [col, "count"]
    plot_df["percent"] = (plot_df["count"] / len(df) * 100).round(1)
    plt.figure(figsize=(12, max(4, 0.45 * len(plot_df))))
    ax = sns.barplot(data=plot_df, x="count", y=col, palette="Set2")
    ax.set_title(title)
    ax.set_xlabel("Jumlah review")
    ax.set_ylabel("")
    for i, row in plot_df.iterrows():
        ax.text(row["count"], i, f" {row['count']:,} ({row['percent']:.1f}%)", va="center")
    plt.tight_layout()
    save_current_figure(save_as)
    plt.show()
    return plot_df

def plot_time_trend(df, title, freq="M", save_as=None):
    temp = df.dropna(subset=["review_date_parsed"]).copy()
    if temp.empty:
        print(f"Tanggal tidak cukup untuk trend: {title}")
        return pd.DataFrame(columns=["period", "review_count"])
    temp["period"] = temp["review_date_parsed"].dt.to_period(freq).dt.to_timestamp()
    trend = temp.groupby("period").size().reset_index(name="review_count")
    plt.figure(figsize=(14, 5))
    ax = sns.lineplot(data=trend, x="period", y="review_count", marker="o")
    ax.set_title(title)
    ax.set_xlabel("Periode")
    ax.set_ylabel("Jumlah review")
    plt.xticks(rotation=45)
    plt.tight_layout()
    save_current_figure(save_as)
    plt.show()
    return trend

def plot_year_trend(df, title, save_as=None):
    temp = df.dropna(subset=["review_date_parsed"]).copy()
    if temp.empty:
        print(f"Tanggal tidak cukup untuk trend tahunan: {title}")
        return pd.DataFrame(columns=["year", "review_count"])
    temp["year"] = temp["review_date_parsed"].dt.year
    trend = temp.groupby("year").size().reset_index(name="review_count")
    plt.figure(figsize=(14, 5))
    ax = sns.barplot(data=trend, x="year", y="review_count", color="#4C78A8")
    sns.lineplot(data=trend, x=np.arange(len(trend)), y="review_count", marker="o", color="#F58518", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Tahun")
    ax.set_ylabel("Jumlah review")
    ax.tick_params(axis="x", rotation=45)
    for i, row in trend.iterrows():
        if row["review_count"] == trend["review_count"].max():
            ax.text(i, row["review_count"], f" {row['review_count']:,}", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    save_current_figure(save_as)
    plt.show()
    return trend

def plot_text_length(df, title, save_as=None):
    temp = df[df["text_review"].str.strip().ne("")].copy()
    if temp.empty:
        print(f"Tidak ada teks untuk {title}")
        return pd.DataFrame()
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    sns.histplot(temp["char_len"], bins=50, kde=True, ax=axes[0], color="#2A9D8F")
    axes[0].set_title(f"Panjang karakter - {title}")
    axes[0].set_xlabel("Jumlah karakter")
    axes[0].axvline(temp["char_len"].median(), color="#264653", linestyle="--", label=f"Median: {temp['char_len'].median():.0f}")
    axes[0].axvline(20, color="#E76F51", linestyle=":", label="Ambang 20 karakter")
    axes[0].legend()
    sns.histplot(temp["word_count"], bins=50, kde=True, ax=axes[1], color="#E76F51")
    axes[1].set_title(f"Jumlah kata - {title}")
    axes[1].set_xlabel("Jumlah kata")
    axes[1].axvline(temp["word_count"].median(), color="#264653", linestyle="--", label=f"Median: {temp['word_count'].median():.0f}")
    axes[1].legend()
    plt.tight_layout()
    save_current_figure(save_as)
    plt.show()
    return temp[["review_id", "char_len", "word_count"]]

# Output utama untuk naskah/subbab EDA: memakai data hasil preprocessing.
report_platform_counts = plot_top_counts(clean_std, "platform", "Sebaran Ulasan per Platform OTA", save_as="fig_platform.png")
report_hotel_counts = plot_top_counts(clean_std, "hotel_name", "Sebaran Ulasan per Cabang Hotel Santika", top_n=12, save_as="fig_cabang.png")
report_year_counts = plot_year_trend(clean_std, "Tren Jumlah Ulasan per Tahun", save_as="fig_tahun.png")
report_text_lengths = plot_text_length(clean_std, "Clean Preprocessing", save_as="fig_panjang.png")

display(Markdown("### Tabel angka siap pakai untuk naskah"))
display(Markdown("#### Platform"))
display(report_platform_counts)
display(Markdown("#### Cabang hotel"))
display(report_hotel_counts)
display(Markdown("#### Tahun"))
display(report_year_counts)

report_key_numbers = pd.DataFrame([
    {"metric": "raw_rows", "value": len(raw_std)},
    {"metric": "clean_rows", "value": len(clean_std)},
    {"metric": "removed_rows", "value": len(raw_std) - len(clean_std)},
    {"metric": "min_date", "value": clean_std["review_date_parsed"].min()},
    {"metric": "max_date", "value": clean_std["review_date_parsed"].max()},
    {"metric": "median_char_len", "value": clean_std.loc[clean_std["text_review"].str.strip().ne(""), "char_len"].median()},
    {"metric": "median_word_count", "value": clean_std.loc[clean_std["text_review"].str.strip().ne(""), "word_count"].median()},
])
display(Markdown("#### Ringkasan kunci"))
display(report_key_numbers)

# Plot tambahan untuk eksplorasi, bila ingin membandingkan raw dan clean secara visual.
plot_top_counts(raw_std, "platform", "Raw Merge - Distribusi Platform")
plot_top_counts(raw_std, "hotel_name", "Raw Merge - Distribusi Hotel", top_n=12)
plot_time_trend(raw_std, "Raw Merge - Trend Review Bulanan")


## 6. Perbandingan hasil merge vs preprocessing

In [ ]:
comparison_df = pd.DataFrame([
    {"metric": "rows", "raw_merge": len(raw_std), "clean_preprocessing": len(clean_std)},
    {"metric": "unique_review_id", "raw_merge": raw_std["review_id"].nunique(), "clean_preprocessing": clean_std["review_id"].nunique()},
    {"metric": "unique_platform", "raw_merge": raw_std["platform"].replace("", np.nan).nunique(), "clean_preprocessing": clean_std["platform"].replace("", np.nan).nunique()},
    {"metric": "unique_hotel", "raw_merge": raw_std["hotel_name"].replace("", np.nan).nunique(), "clean_preprocessing": clean_std["hotel_name"].replace("", np.nan).nunique()},
    {"metric": "avg_char_len", "raw_merge": raw_std["char_len"].mean(), "clean_preprocessing": clean_std["char_len"].mean()},
    {"metric": "avg_word_count", "raw_merge": raw_std["word_count"].mean(), "clean_preprocessing": clean_std["word_count"].mean()},
])
comparison_df["difference_clean_minus_raw"] = comparison_df["clean_preprocessing"] - comparison_df["raw_merge"]
display(comparison_df)

raw_ids = set(raw_std["review_id"].replace("", np.nan).dropna())
clean_ids = set(clean_std["review_id"].replace("", np.nan).dropna())
id_comp = pd.DataFrame([
    {"metric": "ID ada di raw", "count": len(raw_ids)},
    {"metric": "ID ada di clean", "count": len(clean_ids)},
    {"metric": "ID bertahan setelah preprocessing", "count": len(raw_ids & clean_ids)},
    {"metric": "ID raw yang hilang setelah preprocessing", "count": len(raw_ids - clean_ids)},
    {"metric": "ID clean yang tidak ada di raw", "count": len(clean_ids - raw_ids)},
])
display(id_comp)

common = raw_std[["review_id", "char_len", "word_count", "text_review"]].merge(
    clean_std[["review_id", "char_len", "word_count", "text_review"]],
    on="review_id",
    suffixes=("_raw", "_clean"),
)

if not common.empty:
    common["char_len_delta"] = common["char_len_clean"] - common["char_len_raw"]
    common["word_count_delta"] = common["word_count_clean"] - common["word_count_raw"]
    display(common[["review_id", "char_len_raw", "char_len_clean", "char_len_delta", "word_count_raw", "word_count_clean", "word_count_delta"]].head())

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    sns.histplot(common["char_len_delta"], bins=50, kde=True, ax=axes[0], color="#457B9D")
    axes[0].set_title("Delta panjang karakter: clean - raw")
    axes[0].set_xlabel("Delta karakter")
    sns.histplot(common["word_count_delta"], bins=50, kde=True, ax=axes[1], color="#F4A261")
    axes[1].set_title("Delta jumlah kata: clean - raw")
    axes[1].set_xlabel("Delta kata")
    plt.tight_layout()
    plt.show()
else:
    print("Tidak ada ID yang sama antara raw dan clean, perbandingan panjang teks dilewati.")


## 7. Wordcloud dan top kata

In [ ]:
try:
    from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
    sastrawi_stopwords = set(StopWordRemoverFactory().get_stop_words())
except Exception:
    sastrawi_stopwords = set()

CUSTOM_STOPWORDS = {
    "yang", "dan", "atau", "dengan", "untuk", "dari", "pada", "dalam", "karena", "jadi",
    "itu", "ini", "ada", "adalah", "saya", "kami", "kita", "mereka", "dia", "nya", "yg",
    "ke", "di", "sih", "deh", "dong", "aja", "saja", "banget", "sekali", "lebih", "kurang",
    "hotel", "santika", "review", "ulasan", "traveloka", "agoda", "tiket",
    "bandung", "bekasi", "bogor", "cirebon", "depok", "tasik", "tasikmalaya", "megacity", "mega", "city",
}
ALL_STOPWORDS = set(STOPWORDS) | sastrawi_stopwords | CUSTOM_STOPWORDS
KEEP_SHORT = {"ac", "tv", "wc"}

def tokenize_text(series, extra_stopwords=None):
    text = " ".join(series.fillna("").astype(str).tolist()).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    stopwords = ALL_STOPWORDS | set(extra_stopwords or [])
    tokens = []
    for token in text.split():
        if token in stopwords:
            continue
        if len(token) <= 2 and token not in KEEP_SHORT:
            continue
        tokens.append(token)
    return tokens

def make_wordcloud(tokens, title, filename=None, max_words=200, colormap="viridis"):
    freq = Counter(tokens)
    if not freq:
        print(f"Tidak ada token untuk wordcloud: {title}")
        return pd.DataFrame(columns=["term", "count"])

    wc = WordCloud(
        width=1400,
        height=700,
        background_color="white",
        max_words=max_words,
        colormap=colormap,
        collocations=False,
        random_state=42,
    ).generate_from_frequencies(freq)

    plt.figure(figsize=(14, 7))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(title, fontsize=18, pad=16)
    plt.tight_layout()
    if filename:
        plt.savefig(OUTPUT_DIR / filename, dpi=150, bbox_inches="tight")
    plt.show()

    return pd.DataFrame(freq.most_common(50), columns=["term", "count"])

raw_tokens = tokenize_text(raw_std["text_review"])
clean_tokens = tokenize_text(clean_std["text_review"])

top_words_raw = make_wordcloud(raw_tokens, "Wordcloud Raw Merge", "wordcloud_raw_merge.png", colormap="crest")
top_words_clean = make_wordcloud(clean_tokens, "Wordcloud Ulasan Keseluruhan", "fig_wordcloud.png", colormap="mako")
make_wordcloud(clean_tokens, "Wordcloud Clean Preprocessing", "wordcloud_clean_preprocessing.png", colormap="mako")

display(Markdown("### Top 30 kata - Raw Merge"))
display(top_words_raw.head(30))
display(Markdown("### Top 30 kata - Clean Preprocessing"))
display(top_words_clean.head(30))


## 8. Wordcloud per platform dan per hotel

In [ ]:
def safe_filename(value):
    value = re.sub(r"[^a-zA-Z0-9]+", "_", str(value).strip().lower())
    return value.strip("_") or "unknown"

def wordcloud_by_category(df, category_col, prefix, top_n=3, min_rows=10):
    categories = df[category_col].replace("", np.nan).dropna().value_counts().head(top_n)
    tables = {}
    for category, count in categories.items():
        subset = df[df[category_col] == category]
        if len(subset) < min_rows:
            continue
        tokens = tokenize_text(subset["text_review"], extra_stopwords=[str(category).lower()])
        filename = f"wordcloud_{prefix}_{safe_filename(category)}.png"
        title = f"Wordcloud {prefix}: {category} ({count:,} review)"
        tables[category] = make_wordcloud(tokens, title, filename=filename, colormap="viridis")
    return tables

platform_word_tables = wordcloud_by_category(clean_std, "platform", "platform", top_n=3)
hotel_word_tables = wordcloud_by_category(clean_std, "hotel_name", "hotel", top_n=6)

for category, table in platform_word_tables.items():
    display(Markdown(f"### Top kata platform: {category}"))
    display(table.head(20))

for category, table in hotel_word_tables.items():
    display(Markdown(f"### Top kata hotel: {category}"))
    display(table.head(20))


## 9. Bigram dan trigram paling sering

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def clean_doc_for_ngram(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def top_ngrams(series, ngram_range=(2, 2), top_n=30, min_df=2):
    docs = series.fillna("").astype(str).map(clean_doc_for_ngram)
    docs = docs[docs.str.len() > 0]
    if docs.empty:
        return pd.DataFrame(columns=["ngram", "count"])
    vectorizer = CountVectorizer(
        stop_words=sorted(ALL_STOPWORDS),
        ngram_range=ngram_range,
        min_df=min_df,
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
    )
    matrix = vectorizer.fit_transform(docs)
    counts = np.asarray(matrix.sum(axis=0)).ravel()
    terms = vectorizer.get_feature_names_out()
    out = pd.DataFrame({"ngram": terms, "count": counts}).sort_values("count", ascending=False)
    return out.head(top_n).reset_index(drop=True)

bigram_clean = top_ngrams(clean_std["text_review"], (2, 2), top_n=30)
trigram_clean = top_ngrams(clean_std["text_review"], (3, 3), top_n=30)

display(Markdown("### Bigram paling sering - Clean Preprocessing"))
display(bigram_clean)
display(Markdown("### Trigram paling sering - Clean Preprocessing"))
display(trigram_clean)

def plot_ngram_table(table, title):
    if table.empty:
        print(f"Tidak ada data: {title}")
        return
    plt.figure(figsize=(12, max(4, 0.4 * len(table))))
    ax = sns.barplot(data=table, x="count", y="ngram", palette="mako")
    ax.set_title(title)
    ax.set_xlabel("Frekuensi")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

plot_ngram_table(bigram_clean, "Top Bigram - Clean Preprocessing")
plot_ngram_table(trigram_clean, "Top Trigram - Clean Preprocessing")


## 10. Simpan output tabel ringkasan ke Google Drive

In [ ]:
summary_df.to_csv(OUTPUT_DIR / "eda_dataset_summary.csv", index=False, encoding="utf-8-sig")
comparison_df.to_csv(OUTPUT_DIR / "eda_raw_vs_clean_comparison.csv", index=False, encoding="utf-8-sig")
id_comp.to_csv(OUTPUT_DIR / "eda_id_comparison.csv", index=False, encoding="utf-8-sig")
report_platform_counts.to_csv(OUTPUT_DIR / "report_platform_counts.csv", index=False, encoding="utf-8-sig")
report_hotel_counts.to_csv(OUTPUT_DIR / "report_hotel_counts.csv", index=False, encoding="utf-8-sig")
report_year_counts.to_csv(OUTPUT_DIR / "report_year_counts.csv", index=False, encoding="utf-8-sig")
report_key_numbers.to_csv(OUTPUT_DIR / "report_key_numbers.csv", index=False, encoding="utf-8-sig")
top_words_raw.to_csv(OUTPUT_DIR / "top_words_raw_merge.csv", index=False, encoding="utf-8-sig")
top_words_clean.to_csv(OUTPUT_DIR / "top_words_clean_preprocessing.csv", index=False, encoding="utf-8-sig")
bigram_clean.to_csv(OUTPUT_DIR / "top_bigram_clean_preprocessing.csv", index=False, encoding="utf-8-sig")
trigram_clean.to_csv(OUTPUT_DIR / "top_trigram_clean_preprocessing.csv", index=False, encoding="utf-8-sig")

for category, table in platform_word_tables.items():
    table.to_csv(OUTPUT_DIR / f"top_words_platform_{safe_filename(category)}.csv", index=False, encoding="utf-8-sig")

for category, table in hotel_word_tables.items():
    table.to_csv(OUTPUT_DIR / f"top_words_hotel_{safe_filename(category)}.csv", index=False, encoding="utf-8-sig")

print("Output tersimpan di:")
print(OUTPUT_DIR)
print("\nFile utama:")
for p in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", p.name)
for p in sorted(OUTPUT_DIR.glob("*.png")):
    print("-", p.name)
